In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from keras import layers
import os
from tensorboard.plugins.hparams import api as hp
from tensorboard import version
from sklearn.model_selection import train_test_split



c:\Users\reziv\Documents\GitHub\Lung_Cancer_Git\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Num GPUs Available:  0
a


In [3]:
df = pd.read_csv('..\\..\\Datasets\\NewDataset\\ProcessedDataset.csv')
print(df.corr()['Severity'].sort_values(ascending=False))

X = df.drop(['Severity'], axis=1)
y = df['Severity']

Severity             1.000000
Air Pollution        0.382265
Alcohol Usage        0.223088
Obesity              0.067149
Passive Smoker       0.052250
Age                  0.035742
Smoking              0.011514
Chest Pain          -0.001281
Genetic Risk        -0.033536
Coughing of Blood   -0.034209
Gender              -0.074792
Lung Disease        -0.101964
Name: Severity, dtype: float64


In [4]:
nums = []

def run(hparams, num, i): #run_dir,
    #with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)
        accuracy = train_test_model(hparams)
        print(accuracy)
        if (i == 0):
            nums.append(accuracy)
        else: 
            nums[num] += accuracy
        if (i == 4):
            average = nums[num]/5
            nums[num] = average
            print(average)
            #tf.summary.scalar(METRIC_ACCURACY, average, step=1)

In [5]:
def train_test_model(hparams):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer=hparams[HP_OPTIMIZER],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=hparams[HP_EPOCHS],
        steps_per_epoch = int(1173/hparams[HP_EPOCHS]),
        verbose = 1,
        callbacks = [
            
        ]
    ) #hp.KerasCallback(direct, hparams), tf.keras.callbacks.TensorBoard(direct)
    _, accuracy = model.evaluate(X_test, y_test)
    #tf.keras.utils.plot_model(model, show_shapes=True, rankdir="LR")
    return accuracy

In [6]:
Permutations = []
for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete(list(range(72, 75, 1)))) #Was 17, 21, 1
    HP_DROPOUT = hp.HParam('dropout', hp.RealInterval(0.0, 0.0)) #hp.RealInterval(0.0, 0.1)
    HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['Adam', 'sgd']))
    HP_EPOCHS = hp.HParam('epochs', hp.Discrete(list(range(52, 55, 1)))) #Was 35, 40, 1
    METRIC_ACCURACY = 'accuracy'
    if (i == 0):
        '''with tf.summary.create_file_writer(direct).as_default():
            hp.hparams_config(
                hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER, HP_EPOCHS],
                metrics=[hp.Metric(METRIC_ACCURACY, display_name='Accuracy')],
            )'''
    session_num = 0
    for num_units in HP_NUM_UNITS.domain.values:
        for dropout_rate in (HP_DROPOUT.domain.min_value, HP_DROPOUT.domain.max_value):
            for optimizer in HP_OPTIMIZER.domain.values:
                for epoch in HP_EPOCHS.domain.values:
                    hparams = {
                        HP_NUM_UNITS: num_units,
                        HP_DROPOUT: dropout_rate,
                        HP_OPTIMIZER: optimizer,
                        HP_EPOCHS: epoch
                    }
                    run_name = "run-%d" %  i + "-" + str(session_num)
                    print('--- Starting trial: %s' % run_name)
                    print({h.name: hparams[h] for h in hparams})
                    Permutations.append({h.name: hparams[h] for h in hparams})
                    run(hparams=hparams, num=session_num, i=i) #direct + "/" + 
                    session_num += 1

for trial in nums:
    trial = trial / 5.0

for i in range(len(nums)):
    print(f"Trial {i}: {nums[i]} ")
Best = max(nums)
BestCombination = Permutations[nums.index(Best)]
print(f"Best trial: {Best}")
print(f"Best combination: {BestCombination}")


--- Starting trial: run-0-0
{'num_units': 72, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 52}
Epoch 1/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3968 - loss: 1.3053
Epoch 2/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5913 - loss: 0.9507
Epoch 3/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6655 - loss: 0.8262
Epoch 4/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6843 - loss: 0.7537
Epoch 5/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7329 - loss: 0.6712
Epoch 6/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7389 - loss: 0.6224
Epoch 7/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.5799
Epoch 8/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7679 - loss: 0.5558
Epoch 9/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7645 - loss: 0.5426
Epoch 10/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8038 - loss: 0.4839
Epoch 11/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

Exception ignored in: <function WeakKeyDictionary.__init__.<locals>.remove at 0x00000203B50E3EC0>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\weakref.py", line 370, in remove
    self = selfref()
           ^^^^^^^^^
KeyboardInterrupt: 


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4019 - loss: 1.4056
Epoch 2/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5802 - loss: 0.9466
Epoch 3/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6263 - loss: 0.8438 
Epoch 4/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6451 - loss: 0.7964
Epoch 5/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6715 - loss: 0.7390
Epoch 6/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6886 - loss: 0.6970
Epoch 7/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7210 - loss: 0.6715
Epoch 8/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7167 - loss: 0.6373
Epoch 9/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7312 - loss: 0.6329
Epoch 10/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7440 - loss: 0.6097
Epoch 11/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7568 - loss: 0.5866
Epoch 12/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7679 - loss: 0.5688

KeyboardInterrupt: 

In [13]:
def train_test_tuned_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(72, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(72, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(72, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(72, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(72, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer= 'Adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=52,
        steps_per_epoch = int(1173/51),
    )
    _, accuracy = model.evaluate(X_test, y_test)
    return accuracy


sum = 0

for i in range(10):
    run_name = "run-%d" % (i+1)
    print('--- Starting trial: %s' % run_name)
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    data = train_test_tuned_model()
    sum += data
    print(data)
    
average = sum / 10

print(average)


--- Starting trial: run-1
Epoch 1/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4718 - loss: 1.1907
Epoch 2/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5589 - loss: 0.9234
Epoch 3/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6416 - loss: 0.7781
Epoch 4/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6570 - loss: 0.7261
Epoch 5/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7167 - loss: 0.6354
Epoch 6/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7381 - loss: 0.6031
Epoch 7/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7577 - loss: 0.5677
Epoch 8/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7594 - loss: 0.5374
Epoch 9/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7986 - loss: 0.4850
Epoch 10/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7995 - loss: 0.4485
Epoch 11/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8276 - loss: 0.3995
Epoch 12/52
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/st